In [0]:
%pip install -U -qqqq mlflow databricks-langchain databricks-agents uv langgraph==0.3.4
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
%pip install typing_extensions==4.11.0


In [0]:
# COMMAND ----------
# MAGIC %pip install typing_extensions==4.11.0


In [0]:

from typing import Any, Generator, Optional, Sequence, Union
import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatAgentChunk, ChatContext

# COMMAND ----------

# Load campaign data
campaign_df = spark.table("solutions_catalog.pih_schema.consideration_data").toPandas()

# COMMAND ----------

# Tools with required docstrings

@tool
def analyze_engagement(campaign_id: int) -> str:
    """Analyze engagement metrics for a campaign by campaign ID."""
    data = campaign_df[campaign_df["Campaign_ID"] == campaign_id]
    if data.empty:
        return f"No data for Campaign ID {campaign_id}."
    engagement = data["Engagement_Rate"].mean()
    brand_index = data["brand_index"].mean()
    digital_index = data["digital_index"].mean()
    return (
        f"Engagement Analysis for Campaign {campaign_id}:\n"
        f"- Avg Engagement Rate: {engagement:.2f}%\n"
        f"- Avg Brand Index: {brand_index:.2f}\n"
        f"- Avg Digital Index: {digital_index:.2f}"
    )

@tool
def analyze_brandlift(campaign_id: int) -> str:
    """Analyze brand lift using brand index for a campaign."""
    data = campaign_df[campaign_df["Campaign_ID"] == campaign_id]
    if data.empty:
        return f"No data for Campaign ID {campaign_id}."
    return f"Brand Lift for Campaign {campaign_id}: {data['brand_index'].mean():.2f}"

@tool
def analyze_reach(campaign_id: int) -> str:
    """Analyze reach index for a campaign."""
    data = campaign_df[campaign_df["Campaign_ID"] == campaign_id]
    if data.empty:
        return f"No data for Campaign ID {campaign_id}."
    return f"Reach Index for Campaign {campaign_id}: {data['Reach_Index'].mean():.2f}"

@tool
def summarize_impressions(campaign_id: int) -> str:
    """Summarize total impressions for a campaign."""
    data = campaign_df[campaign_df["Campaign_ID"] == campaign_id]
    if data.empty:
        return f"No data for Campaign ID {campaign_id}."
    return f"Total Impressions for Campaign {campaign_id}: {data['Impressions'].sum()}"

@tool
def route_to_agent(question: str) -> str:
    """Route a user question to the correct specialized agent."""
    q = question.lower()
    if "engagement" in q: return "EngagementAgent"
    elif "brand" in q: return "BrandAgent"
    elif "reach" in q: return "ReachAgent"
    elif "conversion" in q or "impression" in q: return "ConversionAgent"
    return "UnknownAgent"

# COMMAND ----------

# Function to build tool-based LangGraph agent
def create_tool_calling_agent(model: LanguageModelLike, tools: Union[ToolNode, Sequence[BaseTool]], system_prompt: Optional[str] = None) -> CompiledStateGraph:
    model = model.bind_tools(tools)

    def should_continue(state: ChatAgentState):
        last_message = state["messages"][-1]
        return "continue" if last_message.get("tool_calls") else "end"

    preprocessor = (
        RunnableLambda(lambda state: [{"role": "system", "content": system_prompt}] + state["messages"])
        if system_prompt else RunnableLambda(lambda state: state["messages"])
    )

    model_runnable = preprocessor | model

    def call_model(state: ChatAgentState, config: RunnableConfig):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)
    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {
        "continue": "tools",
        "end": END
    })
    workflow.add_edge("tools", "agent")

    return workflow.compile()

# COMMAND ----------

# ChatAgent Wrapper
class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, messages: list[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))
        return ChatAgentResponse(messages=messages)

    def predict_stream(self, messages: list[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"])

# COMMAND ----------

# Initialize LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# Enable MLflow autologging
mlflow.langchain.autolog()

# Create agents
EngagementAgent = LangGraphChatAgent(create_tool_calling_agent(llm, [analyze_engagement], "You are the EngagementAgent."))
BrandAgent = LangGraphChatAgent(create_tool_calling_agent(llm, [analyze_brandlift], "You are the BrandAgent."))
ReachAgent = LangGraphChatAgent(create_tool_calling_agent(llm, [analyze_reach], "You are the ReachAgent."))
ConversionAgent = LangGraphChatAgent(create_tool_calling_agent(llm, [summarize_impressions], "You are the ConversionAgent."))
CampaignAnalyst = LangGraphChatAgent(create_tool_calling_agent(llm, [route_to_agent], "You are the CampaignAnalyst. Route questions to the correct child agent."))

# COMMAND ----------

# 🔍 Example: Ask parent agent to route
question = "Give me the engagement analysis for campaign ID 5299445"
messages = [ChatAgentMessage(role="user", content=question)]
response = CampaignAnalyst.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

# 🔍 Then call EngagementAgent directly
response = EngagementAgent.predict(messages=[ChatAgentMessage(role="user", content=question)])
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")


User asks a question (e.g., "What’s the engagement for campaign ID 123456?")

CampaignAnalyst routes the question using the route_to_agent tool.

It checks for keywords like engagement, brand, reach, or impression.

Based on routing:

EngagementAgent is invoked.

It uses analyze_engagement(campaign_id) to analyze relevant columns.

The selected agent returns a summary response based on filtered data from the Spark table.

Modular: Each agent handles a narrow concern (separation of concerns).

Scalable: You can easily plug in more agents (e.g., BudgetAgent).

Explainable: Tool-based logic makes it clear how data is interpreted.

Trackable: MLflow can log every step in the decision and analysis flow.